<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/06-cross-validation.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 6 — Cross-Validation and Leakage

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/cross-validation-and-leakage/).

How an honest estimate of a model's error is built, and five ways it quietly stops being honest.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def make_subscribers(n, seed=0):
    rng = np.random.default_rng(seed)
    hours = rng.uniform(0, 10, n)
    months = rng.uniform(1, 36, n)
    p = sigmoid(-3.0 + 0.6 * hours + 0.05 * months)
    return np.column_stack([hours, months]), (rng.uniform(size=n) < p).astype(int)

X, y = make_subscribers(100, seed=1)
X_big, y_big = make_subscribers(20000, seed=2)
model = make_pipeline(StandardScaler(), LogisticRegression())
honest = model.fit(X, y).score(X_big, y_big)
print(f'100 subscribers to learn from. Accuracy of that model on 20,000 new ones: {honest:.3f}')

## 1. One split is one roll of the dice

In [ ]:
scores = []
for seed in range(20):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed)
    scores.append(model.fit(Xtr, ytr).score(Xte, yte))
scores = np.array(scores)
print(f'20 different 70/30 splits of the same 100 rows: accuracy from {scores.min():.2f} to {scores.max():.2f}')
print(f'mean {scores.mean():.3f}, spread (std) {scores.std():.3f}')

## 2. k-fold: use every row for testing once

In [ ]:
for k in [5, 10, 100]:
    cv = KFold(n_splits=k, shuffle=True, random_state=0)
    s = cross_val_score(model, X, y, cv=cv)
    label = 'leave-one-out' if k == 100 else f'{k}-fold'
    print(f'{label:<14} mean {s.mean():.3f}  std across folds {s.std():.3f}  ({k} models trained)')

In [ ]:
# How stable is the k-fold estimate itself? Repeat it with different shufflings.
means = [cross_val_score(model, X, y, cv=KFold(5, shuffle=True, random_state=s)).mean() for s in range(20)]
print(f'5-fold mean, over 20 different shufflings: {min(means):.3f} to {max(means):.3f}')

## 3. Stratify when a class is rare

In [ ]:
rng = np.random.default_rng(5)
keep = (y_big == 1) | (rng.uniform(size=len(y_big)) < 0.08)
X_rare, y_rare = X_big[keep][:300], y_big[keep][:300]
print(f'{len(y_rare)} rows, {np.sum(y_rare == 0)} leavers')
for name, cv in [('plain KFold', KFold(5, shuffle=True, random_state=1)),
                 ('StratifiedKFold', StratifiedKFold(5, shuffle=True, random_state=1))]:
    counts = [int(np.sum(y_rare[test] == 0)) for _, test in cv.split(X_rare, y_rare)]
    print(f'{name:<16} leavers per test fold: {counts}')

## 4. Leak one: choosing features before splitting

In [ ]:
rng = np.random.default_rng(0)
n, p = 50, 1000
X_noise = rng.normal(size=(n, p))             # a thousand features of pure noise
y_coin = rng.integers(0, 2, n)                # a label that is a coin flip

# The wrong way: pick the 20 features most correlated with the label, using ALL the rows, then cross-validate
corr = np.abs([np.corrcoef(X_noise[:, j], y_coin)[0, 1] for j in range(p)])
top = np.argsort(corr)[-20:]
leaky = cross_val_score(LogisticRegression(), X_noise[:, top], y_coin, cv=KFold(5, shuffle=True, random_state=0))
print(f'select features first, then cross-validate: accuracy {leaky.mean():.2f}')

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# The right way: the selection is a step inside the model, so it only ever sees training folds
honest_pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
clean = cross_val_score(honest_pipe, X_noise, y_coin, cv=KFold(5, shuffle=True, random_state=0))
print(f'select features inside each fold          : accuracy {clean.mean():.2f}')
print(f'a coin flip                               : accuracy 0.50')

## 5. Leak two: preprocessing fitted on everything

In [ ]:
# Scaling is the mild case. Imputing missing values with the mean, or encoding a category by its average label, leak more.
Xm = X.copy()
Xm[rng.uniform(size=Xm.shape) < 0.3] = np.nan            # 30% of values missing

from sklearn.impute import SimpleImputer

outside = SimpleImputer().fit_transform(Xm)              # the imputer has seen every row, test rows included
s_outside = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), outside, y,
                            cv=KFold(5, shuffle=True, random_state=0)).mean()
s_inside = cross_val_score(make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression()), Xm, y,
                           cv=KFold(5, shuffle=True, random_state=0)).mean()
print(f'imputer fitted outside the folds: {s_outside:.3f}   inside: {s_inside:.3f}   (small here; the habit is what matters)')

## 6. Leak three: the same customer on both sides

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(11)
customers, months_each = 150, 6
cust_id = np.repeat(np.arange(customers), months_each)                 # one row per customer per month
loyalty = rng.normal(0, 2.0, customers)                                # something about each customer we never measured
usual_hours = rng.uniform(0, 10, customers)
start_tenure = rng.uniform(1, 30, customers)
hours = usual_hours[cust_id] + rng.normal(0, 0.05, len(cust_id))        # this month's hours, close to their usual
tenure = start_tenure[cust_id] + np.tile(np.arange(months_each), customers) / 12   # in years, ticking up each month
p = sigmoid(-3 + 0.6 * usual_hours + 0.05 * start_tenure + loyalty)
renewed = (rng.uniform(size=customers) < p).astype(int)[cust_id]       # ONE outcome per customer, copied to all six rows
Xg = np.column_stack([hours, tenure])

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))
shuffled = cross_val_score(knn, Xg, renewed, cv=KFold(5, shuffle=True, random_state=0)).mean()
grouped = cross_val_score(knn, Xg, renewed, cv=GroupKFold(5), groups=cust_id).mean()
print(f'{customers} customers x {months_each} months = {len(cust_id)} rows')
print(f'rows shuffled across folds : accuracy {shuffled:.3f}')
print(f'customers kept together    : accuracy {grouped:.3f}')
lr_grouped = cross_val_score(model, Xg, renewed, cv=GroupKFold(5), groups=cust_id).mean()
print(f'logistic regression, customers kept together: {lr_grouped:.3f}')

## 7. Leak four: shuffling time

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(3)
weeks = np.arange(104)                                   # two years of weekly demand
demand = 200 + 2.5 * weeks + 30 * np.sin(2 * np.pi * weeks / 52) + rng.normal(0, 15, 104)
Xt = np.column_stack([np.sin(2 * np.pi * weeks / 52), np.cos(2 * np.pi * weeks / 52)])   # season only, no trend

def mae_scores(cv):
    return -cross_val_score(LinearRegression(), Xt, demand, cv=cv, scoring='neg_mean_absolute_error').mean()

print(f'shuffled 5-fold          : mean absolute error {mae_scores(KFold(5, shuffle=True, random_state=0)):6.1f} units')
print(f'forward in time (5 splits): mean absolute error {mae_scores(TimeSeriesSplit(5)):6.1f} units')

## 8. Leak five: a feature that knows the answer

In [ ]:
X_s, y_s = make_subscribers(2000, seed=8)
rng = np.random.default_rng(8)
# "refund_issued" is recorded when a subscriber cancels. It is written AFTER the outcome.
refund = np.where(y_s == 0, rng.uniform(size=2000) < 0.8, rng.uniform(size=2000) < 0.02).astype(int)
X_leak = np.column_stack([X_s, refund])

cv = StratifiedKFold(5, shuffle=True, random_state=0)
print(f'hours and tenure            : accuracy {cross_val_score(model, X_s, y_s, cv=cv).mean():.3f}')
print(f'plus refund_issued          : accuracy {cross_val_score(model, X_leak, y_s, cv=cv).mean():.3f}')

## 9. Leak six: tuning on the test set

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

X_t, y_t = make_subscribers(150, seed=21)
Xtr, Xte, ytr, yte = train_test_split(X_t, y_t, test_size=0.3, random_state=0)
grid = {'svc__C': np.logspace(-2, 3, 12), 'svc__gamma': np.logspace(-3, 1, 9)}
svm = make_pipeline(StandardScaler(), SVC())

# Pick the best of 108 settings by its score on the TEST set
best_test, best_cfg = -1, None
for C in grid['svc__C']:
    for g in grid['svc__gamma']:
        s = svm.set_params(svc__C=C, svc__gamma=g).fit(Xtr, ytr).score(Xte, yte)
        if s > best_test:
            best_test, best_cfg = s, (C, g)
chosen_on_test = svm.set_params(svc__C=best_cfg[0], svc__gamma=best_cfg[1]).fit(Xtr, ytr)
print(f'best of 108 settings, chosen on the {len(yte)} test rows: {best_test:.3f}')
print(f'  ...and what that model actually does on 20,000 fresh subscribers: {chosen_on_test.score(X_big, y_big):.3f}')

# The honest version: choose on training folds, score once on the test set
search = GridSearchCV(svm, grid, cv=5).fit(Xtr, ytr)
print(f'chosen by cross-validation on training rows, scored once on test: {search.score(Xte, yte):.3f}')
print(f'  ...and on 20,000 fresh subscribers: {search.best_estimator_.score(X_big, y_big):.3f}')

In [ ]:
# Nested cross-validation: the whole search is the model, and it is cross-validated as one thing
nested = cross_val_score(GridSearchCV(svm, grid, cv=5), X_t, y_t, cv=5)
print(f'nested cross-validation: {nested.mean():.3f} ± {nested.std():.3f}')

## 10. What the number means

In [ ]:
for n in [50, 200, 800]:
    estimates, actuals = [], []
    for s in range(20):                                   # twenty different datasets of this size
        Xn, yn = make_subscribers(n, seed=100 + s)
        estimates.append(cross_val_score(model, Xn, yn, cv=KFold(5, shuffle=True, random_state=0)).mean())
        actuals.append(model.fit(Xn, yn).score(X_big, y_big))
    print(f'n = {n:>3}: 5-fold estimate {np.mean(estimates):.3f} (varies by ±{np.std(estimates):.3f} between datasets)'
          f'   actual accuracy of the model trained on all n rows {np.mean(actuals):.3f}')

## 11. The one line you would ship

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0)
scores = cross_val_score(model, X, y, cv=cv)
print(f'{len(scores)} folds: accuracy {scores.mean():.3f} ± {scores.std():.3f}')

## Try it yourself

1. In section 4, change the number of noise features from 1,000 to 20. How much does the leaky estimate fall? Why?
2. In section 6, set `loyalty` to zero. Do the two estimates agree now, and what does that tell you about when grouping matters?
3. In section 9, change the test size to 0.1. How much more optimistic does "chosen on the test set" become?